In [ ]:

"""
可以通过在实例化类时指定一个代理字典传递到CybORG中使代理采取行动。每当调用step函数时，代理将轮流执行它们的操作。
"""
from CybORG import CybORG
from CybORG.Simulator.Scenarios import ICSNetworkGenerator
from CybORG.Agents.Wrappers import EnterpriseMAE

#实例化三个代理
from CybORG.Agents import RandomAgent,EnterpriseGreenAgent,FiniteStateRedAgent,BlueReactRemoveAgent

#导入关键的RLLib库
from ray.tune import register_env
from ray.rllib.algorithms.ppo import PPOConfig,PPO
from ray.rllib.algorithms.dqn import DQNConfig,DQN
from ray.rllib.policy.policy import PolicySpec

"""
虽然CybORG使用OpenAI gym API，但它不是通过调用gym.make（）来运行的。
它必须通过调用 环境创建器构造函数 手动实例化。
构造函数有两个强制性的字符串参数：一个模式类型，它指定将在后台使用哪个引擎，
                            以及用于定义网络布局和代理操作空间的场景的类
"""

#用字符串名称注册一个自定义环境创建器函数 返回一个env实例
def env_creator_cc4(env_config: dict):
    sg=ICSNetworkGenerator(
        blue_agent_class=BlueReactRemoveAgent,
        green_agent_class=EnterpriseGreenAgent,
        red_agent_class=FiniteStateRedAgent,
        steps=100
    )
    cyborg=CybORG(scenario_generator=sg)
    env=EnterpriseMAE(env=cyborg)
    return env
#为了让它与tune.register_env（）一起工作，使用带有lambda函数的自定义env。
register_env(name='cc4', env_creator=lambda config: env_creator_cc4(config))
env = env_creator_cc4({})

NUM_AGENTS = 3
POLICY_MAP = {f"blue_agent_{i}": f"Agent{i}" for i in range(NUM_AGENTS)}

def policy_mapper(agent_id, episode, worker, **kwargs):
    return POLICY_MAP[agent_id]

algo_config = (
    PPOConfig()
    .environment(env="cc4")
    .debugging(logger_config={"logdir":"logs/test_Example", "type":"ray.tune.logger.TBXLogger"})
    .env_runners(sample_timeout_s=300)
    .multi_agent(policies={
        ray_agent: PolicySpec(
            policy_class=None,
            observation_space=env.observation_space(cyborg_agent),
            action_space=env.action_space(cyborg_agent),
            config={"gamma": 0.85},
        ) for cyborg_agent, ray_agent in POLICY_MAP.items()
    },
    policy_mapping_fn=policy_mapper
))


algo=algo_config.build()
for i in range(1000):
    trainer = algo.train()
    print(f"Iteration {i+1} completed")
algo.save('res')

#python -m tensorboard.main --logdir=F:\cage-challenge-4-main\mycode\logs\PPO_Example